# ORF307 Final Exam Code

In [12]:
import numpy as np
import cvxpy as cp

## Problem 1: Minimum Cost Flow

In [13]:
# Problem 1 Setup
b = np.array([10, 20, 5, 15, 10, 15])
d = np.array([15, 10, 10, 20, 5, 15])
R = b - d
print("R =", R)

arcs = [(1,2),(2,4),(2,6),(3,4),(3,5),(4,5),(5,1),(6,3)]
c = np.array([2.5,1.8,3.1,1.2,2.0,0.9,4.0,1.5])
m = len(arcs)

R = [-5 10 -5 -5  5  0]


### Problem 1c

In [14]:
# Problem 1c 
n_nodes = 6
A = np.zeros((n_nodes, m))
for k,(i,j) in enumerate(arcs):
    A[i-1, k] =  1
    A[j-1, k] = -1

x = cp.Variable(m, nonneg=True)

objective = cp.Minimize(c @ x)
constraints = [A @ x == R]

prob = cp.Problem(objective, constraints)
prob.solve()

print("Optimal cost Z* =", prob.value)
for (i,j),val in zip(arcs, x.value):
    print(f"x_{i}{j} = {val:.6g}")


Optimal cost Z* = 51.99999998325934
x_12 = 0
x_24 = 5
x_26 = 5
x_34 = 1.96174e-09
x_35 = 0
x_45 = 0
x_51 = 5
x_63 = 5


### Problem 1e

In [15]:
# Problem 1e
x2 = cp.Variable(m, nonneg=True)
constraints2 = [A @ x2 == R]
constraints2 += [x2[1] <= 4]

prob2 = cp.Problem(cp.Minimize(c @ x2), constraints2)
prob2.solve(solver=cp.ECOS)

print("New optimal cost Z** =", prob2.value)
for (i,j),val in zip(arcs, x2.value):
    print(f"x_{i}{j} = {val:.6g}")


New optimal cost Z** = 55.99999995928617
x_12 = 0
x_24 = 4
x_26 = 6
x_34 = 1
x_35 = 0
x_45 = 0
x_51 = 5
x_63 = 6


The total cost changes because the added restriction alters the feasible set of flows rather than the cost parameters themselves. In the original problem, the optimal solution routed flow through the lowest-cost available arcs. When the new constraint is imposed, at least one of these low-cost arcs becomes unavailable or capacity-constrained, so the original flow is no longer feasible. To satisfy the same supply and demand conditions, the optimizer must reroute flow through alternative paths with higher per-unit costs. Consequently, additional constraints become binding and the marginal cost of sending flow increases. Since arc costs and node requirements remain unchanged, the increase in total cost is entirely due to this forced reallocation onto more expensive routes.

### Problem 1f

In [16]:
# Problem 1f
yi = cp.Variable(n_nodes)
dual_constraints = []
for (i,j), cij in zip(arcs, c):
    dual_constraints += [yi[i-1] - yi[j-1] <= cij]

dual_prob = cp.Problem(cp.Maximize(R @ yi), dual_constraints)
dual_prob.solve(solver=cp.ECOS)

print("Dual optimum=", dual_prob.value)
print("y* =", yi.value)

Dual optimum= 51.999999936481956
y* = [-2.38532351  2.56766196 -2.03233804  0.76766196  1.61467649 -0.53233804]


Economically, the dual variable $y_i$ can be interpreted as the shadow price associated with the flow-balance constraint at node $i$, i.e., the marginal value of increasing the net supply $R_i$ at that node by one unit. If $y_i$ is high, additional supply at node $i$ is valuable because it can be routed through the network at relatively low cost. The dual inequalities $y_i - y_j \le c_{ij}$ ensure that the potential difference between two nodes does not exceed the cost of shipping directly between them; otherwise, it would be possible to send flow along arc $(i,j)$ and reduce total cost, contradicting optimality. In this sense, the inequalities enforce that all reduced costs are nonnegative. By complementary slackness, any arc that carries positive flow must satisfy $y_i - y_j = c_{ij}$, meaning the arc is used only when its shipping cost exactly matches the difference in node shadow prices, while arcs with strictly positive reduced cost carry zero flow.

## Problem 2: Multi-Objective Least Squares

### Problem 2c

In [17]:
# Problem 2c
X_feat = np.array([
    [3.0, 12, 7.5],
    [2.0,  8, 5.0],
    [1.5,  6, 4.0],
    [4.0, 18, 9.0],
    [1.0,  4, 2.5],
    [2.5, 10, 7.0],
])

X = np.hstack([np.ones((6,1)), X_feat])

y1 = np.array([15,10,10,20,5,15])
y2 = np.array([16, 9, 8,18,7,16])
y3 = np.array([13,12,10,21,6,15])

beta = np.linalg.solve(X.T @ X, X.T @ ((y1+y2+y3)/3.0))
print("beta* =", beta)

pred = X @ beta
sse = np.sum((pred-y1)**2) + np.sum((pred-y2)**2) + np.sum((pred-y3)**2)
print("Combined SSE =", sse)


beta* = [  1.8        -14.43333333   2.28333333   3.83333333]
Combined SSE = 19.433333333333334


### Problem 2d

In [18]:
# Problem 2d (i)
Xnew_feat = np.array([
    [3.5, 14, 8.0],
    [2.0,  9, 5.5],
    [1.0,  5, 4.5],
    [4.5, 19, 9.5],
    [1.5,  6, 3.0],
    [3.0, 11, 7.5],
], dtype=float)
Xnew = np.hstack([np.ones((6,1)), Xnew_feat])

d_pred_raw = Xnew @ beta
d_pred = np.rint(d_pred_raw).astype(int)
print("d_pred_raw =", d_pred_raw)
print("d_pred (rounded) =", d_pred)

d_pred_raw = [13.91666667 14.56666667 16.03333333 16.65        5.35       12.36666667]
d_pred (rounded) = [14 15 16 17  5 12]


In [19]:
# Problem 2d (ii)
D_pred = int(d_pred.sum())
Delta = D_pred - 75
print("Total predicted demand D_pred =", D_pred)
print("Deficit Δ =", Delta)

Total predicted demand D_pred = 79
Deficit Δ = 4


In [20]:
d_pred_f = d_pred.astype(float)

def solve_mcf(Rvec):
    x = cp.Variable(m, nonneg=True)
    prob = cp.Problem(cp.Minimize(c @ x), [A @ x == Rvec])
    prob.solve(solver=cp.ECOS)
    return prob.value, x.value

costs = {}
flows = {}
for k in range(1,7):
    btilde = b.copy()
    btilde[k-1] += Delta
    Rk = btilde - d_pred_f
    val, xval = solve_mcf(Rk)
    costs[k] = val
    flows[k] = xval

print("Costs by deficit-assignment center:")
for k in range(1,7):
    print(f"k={k}: cost={costs[k]:.6g}")

best_k = min(costs, key=costs.get)
print("\nBest k =", best_k, "with cost", costs[best_k])
print("Flows for best k:")
for (i,j),val in zip(arcs, flows[best_k]):
    print(f"x_{i}{j} = {val:.6g}")

Costs by deficit-assignment center:
k=1: cost=77.4
k=2: cost=67.4
k=3: cost=49
k=4: cost=78.6
k=5: cost=93.4
k=6: cost=55

Best k = 3 with cost 48.99999998310467
Flows for best k:
x_12 = 1
x_24 = 2
x_26 = 4
x_34 = 0
x_35 = 0
x_45 = 0
x_51 = 5
x_63 = 7


The company purchases the total deficit $\Delta$ = 4 and assigns all of it to a single center $k$. For each $k = 1, \dots, 6$, we form the modified supply vector $\tilde{b^{(k)}} = b + \Delta e_k$ and the corresponding net flow requirement vector $R^{(k)} = \tilde{b^{(k)}} - d^{\text{pred}}$. We then re-solve the minimum-cost flow problem from Problem 1 using the same network and arc costs. Comparing the resulting optimal transportation costs across all six cases shows that assigning the entire deficit to center 3 yields the lowest total cost (49.0). Economically, center 3 is well positioned in the network, with relatively low-cost outgoing connections and good access to the rest of the system, so placing the additional supply there minimizes the need to route flow through more expensive paths.

## Problem 3: Integer optimization (CVXPY)

In [21]:
# Problem 3 Setup
profits = np.array([5,7,6], dtype=float)
max_units = np.array([4,6,2], dtype=float)

hours = np.array([
    [3,4,2],
    [4,6,2],
    [2,5,3],
], dtype=float)
avail = np.array([30,40,35], dtype=float)

M = 1e3

In [23]:
# Problem 1b

x = cp.Variable(3, integer=True)
y = cp.Variable(3, boolean=True)
f = cp.Variable(3, boolean=True)

constraints = [
    x >= 0,
    x <= cp.multiply(max_units, y),
    cp.sum(y) <= 2,
    cp.sum(f) == 1,
]

for i in range(3):
    constraints += [hours[i,:] @ x <= avail[i] + M*(1 - f[i])]

prob = cp.Problem(cp.Maximize(profits @ x), constraints)

prob.solve(solver=cp.SCIPY)

print("Z1* =", prob.value)
print("chosen factory f =", f.value)
print("product selected y =", y.value)
print("quantities x =", x.value)


Z1* = 55.0
chosen factory f = [0. 0. 1.]
product selected y = [1. 1. 0.]
quantities x = [ 4.  5. -0.]


In [ ]:
# Problem 1c

x2 = cp.Variable(3, integer=True)
y2 = cp.Variable(3, boolean=True)
f2 = cp.Variable(3, boolean=True)

constraints2 = [
    x2 >= 0,
    x2 <= cp.multiply(max_units, y2),
    cp.sum(f2) == 1,
]
for f in range(3):
    constraints2 += [hours[f,:] @ x2 <= avail[f] + M*(1 - f2[f])]

prob_c = cp.Problem(cp.Maximize(profits @ x2), constraints2)
prob_c.solve(solver=cp.SCIPY)

print("Z2* =", prob_c.value)
print("chosen factory f =", f2.value)
print("product selected y =", y2.value)
print("quantities x =", x2.value)
print("Z1* - Z2* =", prob.value - prob_c.value)


Z2* = 59.999999999999986
chosen factory y = [3.55271368e-15 0.00000000e+00 1.00000000e+00]
product selected z = [1. 1. 1.]
quantities x = [4. 4. 2.]
Z1* - Z2* = -4.999999999999986


In part (c), the restriction that at most two products may be produced is relaxed so that all three products can be produced simultaneously. The increase in the optimal objective value from $Z_1^* = 55$ to $Z_2^*=60$ indicates that the original constraint was binding and economically costly. Under the relaxed model, the firm is able to exploit available factory capacity more efficiently by producing a combination of products that yields higher total profit. In particular, the third product, which was previously excluded due to the “at most two products” constraint, generates positive marginal profit and can be produced without violating the factory time constraint. The difference $Z_1^*-Z_2^*$ = -5 therefore measures the opportunity cost of the product-selection restriction: it represents the amount of profit the firm forgoes in order to comply with the original policy limiting product variety. In summary, the difference shows that the relaxation increases the maximum achievable profit by 5 (the original restriction costs the firm 5 in profit relative to the relaxed model).

In [ ]:
# Problem 3d

x3 = cp.Variable(3, integer=True)
y3 = cp.Variable(3, boolean=True)
f3 = cp.Variable(3, boolean=True)

constraints3 = [
    x3 >= 0,
    x3 <= cp.multiply(max_units, y3),
    cp.sum(y3) <= 2,
    cp.sum(f3) == 1,
    y3[1] <= y3[2],
]
for f in range(3):
    constraints3 += [hours[f,:] @ x3 <= avail[f] + M*(1 - f3[f])]

prob_d = cp.Problem(cp.Maximize(profits @ x3), constraints3)
prob_d.solve(solver=cp.SCIPY)

print("Z3* =", prob_d.value)
print("chosen factory f =", f3.value)
print("product selected y =", y3.value)
print("quantities x =", x3.value)
print("Z1* - Z3* =", prob.value - prob_d.value)


Z3* = 54.00000000000001
chosen factory y = [ 0.  1. -0.]
product selected z = [-0.  1.  1.]
quantities x = [-0.  6.  2.]
Z1* - Z3* = 0.9999999999999929


Under the conditional policy $z_2 \leq z_3$, the optimal profit decreases from $Z_1^*=55$ to $Z_2^*=54$. The difference $Z_1^*-Z_2^* = 1$ quantifies the cost of enforcing the policy: requiring Product 3 to be produced whenever Product 2 is selected restricts the feasible product mix and prevents the firm from choosing the most profitable combination available in the original problem. As a result, production capacity must be reallocated toward a less profitable mix, reducing the maximum achievable profit by 1.